# Data Splitting

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from configurations import TRAIN_PATH, TEST_PATH, TARGET, DROP_FEATURES

"""Load data from the paths specified in the configuration"""
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (26729, 10)
Test shape: (11456, 8)


In [2]:
"""
Define the target variable (y) and input features (X).
Drop the columns that will not be used during model training 
(the target itself, the target subtype, and the unique identifier).
"""
y = train_df[TARGET]

drop_cols = DROP_FEATURES + [TARGET]
X = train_df.drop(columns=[col for col in drop_cols if col in train_df.columns])

print("Selected features:", list(X.columns))

Selected features: ['Name', 'DateTime', 'AnimalType', 'SexuponOutcome', 'AgeuponOutcome', 'Breed', 'Color']


In [3]:
"""
Initialize and execute the Stratified K-Fold cross-validation split.
Stratification ensures that the target variable class proportions 
are preserved across all folds.
"""
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_df['fold'] = -1

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    train_df.loc[val_idx, 'fold'] = fold_idx

print("Number of rows per fold:")
print(train_df['fold'].value_counts())

Number of rows per fold:
fold
3    5346
2    5346
1    5346
0    5346
4    5345
Name: count, dtype: int64


In [4]:
"""
Check the distribution of the target variable.
Compare the percentage proportions of the target classes between the 
full dataset and fold 0 to ensure the stratified split worked correctly.
"""
fold_0_dist = train_df[train_df['fold'] == 0]['OutcomeType'].value_counts(normalize=True) * 100
total_dist = train_df['OutcomeType'].value_counts(normalize=True) * 100

check_df = pd.DataFrame({'Total %': total_dist, 'Fold 0 %': fold_0_dist})
print(check_df)

                   Total %   Fold 0 %
OutcomeType                          
Adoption         40.289573  40.273101
Transfer         35.250103  35.260007
Return_to_owner  17.905646  17.919940
Euthanasia        5.817651   5.817434
Died              0.737027   0.729517


In [5]:
"""
Save the training dataset extended with the 'fold' column in CSV format.
Subsequent notebooks will load this file for model training and testing.
"""
train_df.to_csv('train_folds.csv', index=False)
print("The 'train_folds.csv' has been saved successfully!")

The 'train_folds.csv' has been saved successfully!
